In [ ]:
from pyomo.environ import *
import matplotlib.pyplot as plt

In [ ]:
# Create the model
model = ConcreteModel()

In [ ]:
# Decision variables
# x1 = number of cream cake batches
# x2 = number of chocolate cake batches
model.x1 = Var(domain=NonNegativeIntegers)
model.x2 = Var(domain=NonNegativeIntegers)

In [ ]:
# Objective function: Maximise z = x1 + 3*x2
model.obj = Objective(expr=model.x1 + 3*model.x2, sense=maximize)

In [ ]:
# Use a ConstraintList to hold all constraints
model.constraints = ConstraintList()

# 1) Cream cake batches ≤ 40
model.constraints.add(model.x1 <= 40)
# 2) Chocolate cake batches ≤ 60
model.constraints.add(model.x2 <= 60)
# 3) At least 10 batches of chocolate cake
model.constraints.add(model.x2 >= 10)
# 4) Total number of batches (cream + chocolate) ≥ 20
model.constraints.add(model.x1 + model.x2 >= 20)
# 5) Production time constraint: 3*x1 + 2*x2 ≤ 180
model.constraints.add(3 * model.x1 + 2 * model.x2 <= 180)


In [ ]:
# Solve the model using CBC (or another available MILP solver)
solver = SolverFactory('glpk')  # or any other MILP solver you have installed
results = solver.solve(model, tee=False)

In [ ]:
# Print results
print("Status:", results.solver.status)
print("Termination Condition:", results.solver.termination_condition)
print(f"Optimal number of cream cake batches (x1) = {model.x1.value}")
print(f"Optimal number of chocolate cake batches (x2) = {model.x2.value}")
print(f"Maximum profit (z) = {model.obj.expr()}")

# Data Visualization

In [ ]:
# Generate a list of all feasible integer solutions that satisfy the constraints
feasible_points = []
for x1_test in range(41):    # x1 in 0..40
    for x2_test in range(61):  # x2 in 0..60
        if (x1_test <= 40 and
            x2_test <= 60 and
            x2_test >= 10 and
            (x1_test + x2_test) >= 20 and
            (3 * x1_test + 2 * x2_test) <= 180):
            feasible_points.append((x1_test, x2_test))

# Separate the feasible points into x and y lists
feasible_x1 = [pt[0] for pt in feasible_points]
feasible_x2 = [pt[1] for pt in feasible_points]

# Plot the feasible region and the optimal solution
plt.figure(figsize=(6, 6))
plt.scatter(feasible_x1, feasible_x2, alpha=0.5, label='Feasible Points')
plt.scatter(model.x1.value, model.x2.value, color='red', marker='x',
            s=100, label='Optimal Solution')
plt.xlabel('x1 (Cream Cake Batches)')
plt.ylabel('x2 (Chocolate Cake Batches)')
plt.title('Feasible Solutions and Optimal Solution')
plt.legend()
plt.grid(True)
plt.show()